# v87 tabular学習 — v69_4(base 特徴一式 × PF5バンク tw/self/nbr) + **affine較正PF 15本**追加
入力: v69/artifacts_v69_4/data/train.parquet(転用) + v87/artifacts_v87/data/train_affine_v87.parquet(affine)。
affine候補(afftw/affself/affnbr)と「self/nbr/affine の違い」特徴を id整合で結合。GroupKFold(5) 順方向。出力=v87/artifacts_v87。

In [1]:
# ===== v87: base(v69_4) + gf(v48) + affine(v87) を id整合で"最初から幅広に"確保(hstackのRAMピーク回避) =====
import json, joblib, warnings, gc
from collections import Counter
import numpy as np, pandas as pd
from pathlib import Path
import pyarrow.parquet as pq
from lightgbm import LGBMRegressor, log_evaluation, early_stopping
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import root_mean_squared_error
warnings.filterwarnings("ignore")

PROJECT = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
ART_IN  = PROJECT / "v69" / "artifacts_v69_4"          # ★入力: base parquet(転用)
ART     = PROJECT / "v87" / "artifacts_v87"; ART.mkdir(parents=True, exist_ok=True)   # ★出力: v87
FWD_PQ  = ART_IN / "data" / "train.parquet"
AFF_PQ  = ART / "data" / "train_affine_v87.parquet"     # ★v87 affine(create_affine_v87.py 出力)
USE_BACKWARD = False
MODELS = ART / "models_fwd"; MODELS.mkdir(parents=True, exist_ok=True)
N_SPLITS = 5
ROW_FRAC = 1.0
pp_params = {"alpha": 1.0, "tau": 85, "w_pf": 0.09}

USE_ORIGIN = {"base": True, "pf": True, "beam": True, "self": True, "nbr": True, "zgrad": True}
PREFIXES = [("self", "pf_self"), ("nbr", "pf_nbr"), ("beam", "beam_"), ("pf", "pf_")]
META = {"well", "id", "target", "is_backward"}
def origin_of(c):
    if c == "z_grad": return "zgrad"
    for o, pre in PREFIXES:
        if c.startswith(pre): return o
    return "base"

schema = pq.read_schema(FWD_PQ)
all_feats = [c for c in schema.names if c not in META]
features = [c for c in all_feats if USE_ORIGIN.get(origin_of(c), True)]
print("origin別 全特徴数:", dict(Counter(origin_of(c) for c in all_feats)))

N_FWD = pq.ParquetFile(FWD_PQ).metadata.num_rows

# --- 追加ブロックの列名を先に確定(gf → affine) ---
_GFP = PROJECT / "v48" / "artifacts_v48" / "gf_feature.npy"
gf_cols = ["gf_delta", "gf_std", "gf_vs_struct"] if _GFP.exists() else []
if not gf_cols: print("[warn] gf_feature.npy 無し(先に python v48/gridfilter_feature_v48.py)")
aff_cols = []
if AFF_PQ.exists():
    aff_cols = [c for c in pq.read_schema(AFF_PQ).names if c != "id"]
else:
    print("[warn] train_affine_v87.parquet 無し(先に python v87/create_affine_v87.py)")

feat_all = features + gf_cols + aff_cols
NB = len(features)
print(f"採用 base {NB} + gf {len(gf_cols)} + affine {len(aff_cols)} = {len(feat_all)} 列")
ART.joinpath("features.json").write_text(json.dumps(feat_all))
ART.joinpath("pp_params.json").write_text(json.dumps(pp_params))

# --- 幅広に一括確保(hstack不要=コピーピーク無し) ---
X = np.empty((N_FWD, len(feat_all)), np.float32)

def load_block_into(path, col_list, col_off, nrows):
    """parquet の col_list を X[:, col_off + k] へ列チャンクで書込(欠損列は NaN)。"""
    names = set(pq.read_schema(path).names)
    for j0 in range(0, len(col_list), 24):
        sub = col_list[j0:j0 + 24]; have = [c for c in sub if c in names]
        tb = pq.read_table(path, columns=have) if have else None
        for jj, c in enumerate(sub):
            j = col_off + j0 + jj
            X[:nrows, j] = tb.column(c).to_numpy(zero_copy_only=False).astype(np.float32, copy=False) if c in names else np.nan
        del tb
        if (j0 // 24) % 5 == 4: gc.collect()

# base
load_block_into(FWD_PQ, features, 0, N_FWD)
# gf(v48): np.load、行順=parquet行順(v69_4と同じ)
if gf_cols:
    _GF = np.nan_to_num(np.load(_GFP)[:N_FWD], nan=0.0).astype(np.float32)
    X[:, NB:NB + 3] = _GF; del _GF; gc.collect()
# affine(v87): id整合。行順一致なら positional、ずれてれば reindex
if aff_cols:
    _fid = pq.read_table(FWD_PQ, columns=["id"]).column("id").to_pandas().to_numpy()
    _aid = pq.read_table(AFF_PQ, columns=["id"]).column("id").to_pandas().to_numpy()
    _off = NB + len(gf_cols)
    if len(_aid) == N_FWD and np.array_equal(_aid, _fid):
        load_block_into(AFF_PQ, aff_cols, _off, N_FWD)     # 行順一致=そのまま列チャンク書込
        print("[v87] affine: 行順一致 → positional 書込")
    else:
        _pos = pd.Series(np.arange(len(_aid)), index=_aid).reindex(_fid).to_numpy()  # fwd各行→affine行index
        _ok = np.isfinite(_pos); _posi = np.where(_ok, _pos, 0).astype(np.int64)
        for j0 in range(0, len(aff_cols), 24):
            sub = aff_cols[j0:j0 + 24]; tb = pq.read_table(AFF_PQ, columns=sub)
            for jj, c in enumerate(sub):
                col = tb.column(c).to_numpy(zero_copy_only=False).astype(np.float32)[_posi]
                col[~_ok] = np.nan; X[:, _off + j0 + jj] = col
            del tb
        print(f"[v87] affine: reindex 書込(未マッチ {int((~_ok).sum())} 行=NaN)")
    del _fid, _aid; gc.collect()

# ラベル/群/pp材料
sf = pq.read_table(FWD_PQ, columns=["well", "target", "last_known_tvt", "md_since", "pf_ancc"])
g = sf.column("well").to_pandas().to_numpy()
y = sf.column("target").to_numpy(zero_copy_only=False).astype(np.float32)
lk  = sf.column("last_known_tvt").to_numpy(zero_copy_only=False).astype(np.float64)
mds = sf.column("md_since").to_numpy(zero_copy_only=False).astype(np.float64)
pf_abs = sf.column("pf_ancc").to_numpy(zero_copy_only=False).astype(np.float64)
del sf; gc.collect()
print(f"rows: {N_FWD} | features {len(feat_all)} | wells {pd.unique(g).size} | X {X.nbytes/1e9:.1f} GB")


origin別 全特徴数: {'base': 329, 'beam': 40, 'pf': 238, 'self': 28, 'nbr': 26, 'zgrad': 1}
採用 base 662 + gf 3 + affine 108 = 773 列
[v87] affine: 行順一致 → positional 書込
rows: 3783989 | features 773 | wells 773 | X 11.7 GB


In [2]:
# ===== モデル: LightGBM×3 + CatBoost×3 + XGBoost×3(v20系, GPU) =====
lgb_params = [
    dict(boosting_type="gbdt", num_leaves=255, min_child_samples=15, subsample=0.8, subsample_freq=1,
         colsample_bytree=0.8, reg_lambda=3.0, reg_alpha=0.05, objective="regression", verbose=-1,
         n_jobs=-1, device_type="gpu", gpu_use_dp=False, max_bin=255, learning_rate=0.030, n_estimators=5000, seed=123),
    dict(n_jobs=-1, verbose=-1, reg_alpha=10.788188919840913, subsample=0.47437582748953966, num_leaves=64,
         reg_lambda=95.75401894533888, n_estimators=10000, random_state=0, boosting_type="gbdt",
         learning_rate=0.00934485794382918, colsample_bytree=0.39283351290380497, min_child_weight=0.24081152127177283,
         min_child_samples=40, device="gpu"),
    dict(n_jobs=-1, verbose=-1, reg_alpha=10.788188919840913, subsample=0.47437582748953966, num_leaves=64,
         reg_lambda=95.75401894533888, n_estimators=10000, random_state=29, boosting_type="gbdt",
         learning_rate=0.00934485794382918, colsample_bytree=0.39283351290380497, min_child_weight=0.24081152127177283,
         min_child_samples=40, device="gpu"),
]
cb_params = [
    dict(iterations=8000, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254, loss_function="RMSE",
         task_type="GPU", devices="0", od_type="Iter", od_wait=300, verbose=0, learning_rate=0.020, random_seed=7),
    dict(iterations=8000, depth=7, l2_leaf_reg=2.0, min_data_in_leaf=15, border_count=254, loss_function="RMSE",
         task_type="GPU", devices="0", od_type="Iter", od_wait=300, verbose=0, learning_rate=0.030, random_seed=123),
    dict(iterations=8000, depth=7, l2_leaf_reg=3.0, min_data_in_leaf=15, border_count=192, loss_function="RMSE",
         task_type="GPU", devices="0", od_type="Iter", od_wait=300, verbose=0, learning_rate=0.010, random_seed=999),
]
xgb_params = [
    dict(n_estimators=10000, learning_rate=0.03, max_depth=6, min_child_weight=10, subsample=0.85, colsample_bytree=0.85,
         reg_alpha=0.1, reg_lambda=5.0, objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
         device="cuda", random_state=0, n_jobs=-1, early_stopping_rounds=250),
    dict(n_estimators=10000, learning_rate=0.02, max_depth=8, min_child_weight=8, subsample=0.80, colsample_bytree=0.80,
         reg_alpha=0.05, reg_lambda=4.0, objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
         device="cuda", random_state=1, n_jobs=-1, early_stopping_rounds=250),
    dict(n_estimators=12000, learning_rate=0.02, max_depth=5, min_child_weight=25, subsample=0.75, colsample_bytree=0.75,
         reg_alpha=0.5, reg_lambda=12.0, objective="reg:squarederror", eval_metric="rmse", tree_method="hist",
         device="cuda", random_state=2, n_jobs=-1, early_stopping_rounds=250),
]
VARIANTS = ([(f"lightgbm-{i+1}", "lgb", p) for i, p in enumerate(lgb_params)] +
            [(f"catboost-{i+1}", "cb", p) for i, p in enumerate(cb_params)] +
            [(f"xgboost-{i+1}", "xgb", p) for i, p in enumerate(xgb_params)])
print("variants:", [v[0] for v in VARIANTS])

variants: ['lightgbm-1', 'lightgbm-2', 'lightgbm-3', 'catboost-1', 'catboost-2', 'catboost-3', 'xgboost-1', 'xgboost-2', 'xgboost-3']


In [ ]:
# ===== GroupKFold(5) 学習 → OOF + fold pkl保存(再開可) =====
def fit_one(kind, params, Xtr, ytr, Xva, yva):
    if kind == "lgb":
        m = LGBMRegressor(**params)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="rmse", callbacks=[early_stopping(250), log_evaluation(0)])
    elif kind == "cb":
        m = CatBoostRegressor(**params)
        m.fit(Xtr, ytr, eval_set=(Xva, yva), early_stopping_rounds=250, use_best_model=True, verbose=0)
    else:
        m = XGBRegressor(**params); m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return m
gkf = GroupKFold(n_splits=N_SPLITS); folds = list(gkf.split(X, y, groups=g))
oof = {name: np.zeros(len(y), np.float32) for name, _, _ in VARIANTS}
for name, kind, params in VARIANTS:
    d = MODELS / name; d.mkdir(parents=True, exist_ok=True)
    for k, (tr, va) in enumerate(folds):
        va = va[va < N_FWD]          # ★検証・OOFは順方向行のみ(逆行はtrain専用)
        fp = d / f"fold{k}.pkl"
        if fp.exists(): m = joblib.load(fp); tag = "load"
        else:
            m = fit_one(kind, params, X[tr], y[tr], X[va], y[va]); joblib.dump(m, fp); tag = "fit"
        oof[name][va] = m.predict(X[va])
        print(f"{name} fold{k} [{tag}]: rmse={root_mean_squared_error(y[va], oof[name][va]):.4f}", flush=True)
        del m; gc.collect()
    print(f"== {name} OOF rmse(順方向のみ)={root_mean_squared_error(y[:N_FWD], oof[name][:N_FWD]):.4f}")

# oof 6.87ぐらい

lightgbm-1 fold0 [load]: rmse=6.2063
lightgbm-1 fold1 [load]: rmse=6.8558
lightgbm-1 fold2 [load]: rmse=6.4351
lightgbm-1 fold3 [load]: rmse=6.6372
lightgbm-1 fold4 [load]: rmse=6.5590
== lightgbm-1 OOF rmse(順方向のみ)=6.5421
lightgbm-2 fold0 [load]: rmse=6.3038
lightgbm-2 fold1 [load]: rmse=6.8803
lightgbm-2 fold2 [load]: rmse=6.2229
lightgbm-2 fold3 [load]: rmse=6.4567
lightgbm-2 fold4 [load]: rmse=6.5005
== lightgbm-2 OOF rmse(順方向のみ)=6.4768
lightgbm-3 fold0 [load]: rmse=6.3163
lightgbm-3 fold1 [load]: rmse=6.9051
lightgbm-3 fold2 [load]: rmse=6.3168
lightgbm-3 fold3 [load]: rmse=6.5103
lightgbm-3 fold4 [load]: rmse=6.6139
== lightgbm-3 OOF rmse(順方向のみ)=6.5361
catboost-1 fold0 [load]: rmse=6.3471
catboost-1 fold1 [fit]: rmse=7.0631


In [ ]:
# ===== アンサンブル OOF + apply_pp(submit準拠: pf_res=pf_ancc-lk) → overall RMSE =====
names = [v[0] for v in VARIANTS]
oof_ens = np.mean([oof[n] for n in names], axis=0)[:N_FWD]
w_pf = float(pp_params.get("w_pf", 0.09)); tau = pp_params.get("tau", 85); alpha = float(pp_params.get("alpha", 1.0))
pf_res = pf_abs - lk
def apply_pp_res(res):
    d = res * (1 - w_pf) + pf_res * w_pf
    if tau: d = d * (1.0 - np.exp(-np.maximum(mds, 0.0) / float(tau)))
    return d * alpha
raw = root_mean_squared_error(y[:N_FWD], oof_ens); ppr = root_mean_squared_error(y[:N_FWD], apply_pp_res(oof_ens))
print("=== OOF overall RMSE(全行プール=Kaggle指標) ===")
print(f"  raw(GBMアンサンブル) : {raw:.4f}")
print(f"  + apply_pp           : {ppr:.4f}")
print("  ※ sg は per-well=submitで適用。CV↔LB は ±0.2 床込みで。")

=== OOF overall RMSE(全行プール=Kaggle指標) ===
  raw(GBMアンサンブル) : 6.5892
  + apply_pp           : 6.5974
  ※ sg は per-well=submitで適用。CV↔LB は ±0.2 床込みで。


In [ ]:
# ===== v29 NN(combo) との残差ブレンド + 公平基準(旧v28 GBM×NN を同一行で再計算) =====
V29_ART  = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v29\artifacts_v29")
V29B_ART = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v29\artifacts_v29b")
_step = max(1, int(round(1.0 / ROW_FRAC))) if ROW_FRAC < 0.6 else 1
nnz = np.load(V29B_ART / "nn_oof_combo.npz")
nn_ens = np.mean([nnz[k] for k in nnz.files], axis=0)[::_step]
gbm_old = np.load(V29_ART / "gbm_oof.npy")[::_step]
assert len(nn_ens) == N_FWD == len(gbm_old), (len(nn_ens), N_FWD, len(gbm_old))
def overall_pp(res): return root_mean_squared_error(y[:N_FWD], apply_pp_res(res))
ws2 = np.arange(0.0, 1.001, 0.02)
scs0 = [overall_pp((1 - w2) * gbm_old + w2 * nn_ens) for w2 in ws2]
b0 = int(np.argmin(scs0))
scs = [overall_pp((1 - w2) * oof_ens + w2 * nn_ens) for w2 in ws2]
bi = int(np.argmin(scs))
print(f"公平基準(旧v28GBM×NN, 同一行): 旧GBM単体 {overall_pp(gbm_old):.4f} / blend w_nn={ws2[b0]:.2f} -> {scs0[b0]:.4f}")
print(f"GBM(v36)単体+pp             : {overall_pp(oof_ens):.4f}")
print(f"blend GBM(v36)×NN           : w_nn={ws2[bi]:.2f} -> {scs[bi]:.4f}  (Δ vs 基準 {scs[bi] - scs0[b0]:+.4f})")
print("※ 行はROW_FRAC間引き後の同一行同士。全行の絶対値(6.8711等)とは僅かにずれるがΔは公平。")
np.save(ART / "gbm_v41_oof.npy", oof_ens)
print("saved:", ART / "gbm_v41_oof.npy")


公平基準(旧v28GBM×NN, 同一行): 旧GBM単体 7.0409 / blend w_nn=0.60 -> 6.8699
GBM(v36)単体+pp             : 6.5974
blend GBM(v36)×NN           : w_nn=0.32 -> 6.5004  (Δ vs 基準 -0.3695)
※ 行はROW_FRAC間引き後の同一行同士。全行の絶対値(6.8711等)とは僅かにずれるがΔは公平。
saved: C:\Users\kosaka256\Documents\rogii_claude\v69\artifacts_v69_4\gbm_v41_oof.npy
